In [8]:
import os
from typing import Tuple, Dict, Any, List

import torch
from datasets import Dataset
from transformers import (
                            BertTokenizer,
                            BertForMaskedLM,
                            DataCollatorForLanguageModeling,
                            Trainer,
                            TrainingArguments,
                        )

In [9]:
def get_device() -> torch.device:
    """Return cuda if available else cpu."""
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def set_seed(seed: int = 42) -> None:
    """Set global random seeds (CPU and CUDA)."""
    
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

In [11]:
def build_texts() -> List[str]:
    """Provide raw training texts (replace with your corpus)."""
    
    return [
        "The cat sat on the mat",
        "I love eating pizza",
        "BERT is great for NLP tasks",
        "Transformers are powerful models",
        "Self supervised learning is useful",
        "Masked language modeling helps pretraining",
    ]


def build_dataset(texts: List[str]) -> Dataset:
    """Create a Hugging Face Dataset from raw texts."""
    
    return Dataset.from_dict({"text": texts})


def load_tokenizer(model_name: str = "bert-base-uncased") -> BertTokenizer:
    """Load a tokenizer."""
    
    return BertTokenizer.from_pretrained(model_name)


def tokenize_dataset(
                    dataset: Dataset,
                    tokenizer: BertTokenizer,
                    max_length: int = 24,
                ) -> Dataset:
    """Tokenize the dataset and remove raw text column."""
    
    def _tok(examples):
        return tokenizer(
                        examples["text"],
                        truncation=True,
                        padding="max_length",
                        max_length=max_length,
                    )
        
    return dataset.map(_tok, batched=True, remove_columns=["text"])


def build_collator(
                    tokenizer: BertTokenizer,
                    mlm_probability: float = 0.15,
                ) -> DataCollatorForLanguageModeling:
    
    """Create MLM data collator that dynamically masks tokens."""
    return DataCollatorForLanguageModeling(
                                            tokenizer=tokenizer,
                                            mlm=True,
                                            mlm_probability=mlm_probability,
                                        )

In [16]:
def load_model(model_name: str = "bert-base-uncased", device: torch.device = None) -> BertForMaskedLM:
    """Load a pretrained MLM model and move to device."""
    
    model = BertForMaskedLM.from_pretrained(model_name)
    if device is None:
        device = get_device()
    model.to(device)
    return model


def build_training_args(
    output_dir: str = "./mlm-output",
    num_train_epochs: int = 100,
    per_device_train_batch_size: int = 4,
    learning_rate: float = 5e-5,
    weight_decay: float = 0.01,
    logging_steps: int = 10,
    save_steps: int = 1000,
    save_total_limit: int = 1,
    fp16: bool = None,
) -> TrainingArguments:
    """Create TrainingArguments; fp16 auto-enabled on GPU if not provided."""
    if fp16 is None:
        fp16 = torch.cuda.is_available()
    return TrainingArguments(
        output_dir=output_dir,
        overwrite_output_dir=True,
        num_train_epochs=num_train_epochs,
        per_device_train_batch_size=per_device_train_batch_size,
        learning_rate=learning_rate,
        weight_decay=weight_decay,
        logging_steps=logging_steps,
        save_steps=save_steps,
        save_total_limit=save_total_limit,
        fp16=fp16,
        report_to="none",
    )


def train_model(
    model: BertForMaskedLM,
    tokenized_dataset: Dataset,
    data_collator: DataCollatorForLanguageModeling,
    training_args: TrainingArguments,
) -> None:
    """Train the model using HF Trainer."""
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_dataset,
        data_collator=data_collator,
    )
    trainer.train()

In [17]:
def predict_mask(
    model: BertForMaskedLM,
    tokenizer: BertTokenizer,
    text_with_mask: str,
    device: torch.device = None,
) -> str:
    """Predict the token for the first [MASK] in the input sentence."""
    if device is None:
        device = get_device()

    model.eval()
    inputs = tokenizer(text_with_mask, return_tensors="pt").to(device)

    with torch.no_grad():
        logits = model(**inputs).logits

    mask_pos = (inputs["input_ids"] == tokenizer.mask_token_id).nonzero(as_tuple=True)
    if len(mask_pos[0]) == 0:
        raise ValueError("No [MASK] token found in the input.")
    b_idx = mask_pos[0][0].item()
    s_pos = mask_pos[1][0].item()

    pred_id = logits[b_idx, s_pos].argmax(dim=-1).item()
    return tokenizer.decode([pred_id])


def save_model_safetensors(
    model: BertForMaskedLM,
    tokenizer: BertTokenizer,
    save_dir: str = "./mlm-output/final",
) -> None:
    """
    Save model + tokenizer. safe_serialization=True writes .safetensors.
    """
    os.makedirs(save_dir, exist_ok=True)
    model.save_pretrained(save_dir, safe_serialization=True)
    tokenizer.save_pretrained(save_dir)

In [20]:
def main() -> None:
    os.environ["TOKENIZERS_PARALLELISM"] = "false"

    device = get_device()
    print(f"Using device: {device}")
    set_seed(42)

    # Data
    texts = build_texts()
    raw_ds = build_dataset(texts)
    tokenizer = load_tokenizer()
    tokenized_ds = tokenize_dataset(raw_ds, tokenizer, max_length=24)
    collator = build_collator(tokenizer, mlm_probability=0.15)

    # Model & Train
    model = load_model(device=device)
    train_args = build_training_args(output_dir="./mlm-output")
    train_model(model, tokenized_ds, collator, train_args)

    # Inference
    demo = "I am going to [MASK]"
    prediction = predict_mask(model, tokenizer, demo, device=device)
    print(f"Input : {demo}")
    print(f"Pred  : {prediction}")

    # Save
    save_model_safetensors(model, tokenizer, "./mlm-output/final")
    print("Saved model to ./mlm-output/final (safetensors).")

In [21]:
main()

Using device: cuda


Map:   0%|          | 0/6 [00:00<?, ? examples/s]

Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Detected kernel version 5.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


Step,Training Loss
10,4.129000
20,3.739100
30,2.142800
40,1.720300
50,1.093900
60,0.773400
70,1.261900
80,0.352200
90,0.694500
100,0.352700


Input : I am going to [MASK]
Pred  : bed
Saved model to ./mlm-output/final (safetensors).
